# NutriLens: fine-tune a food recognition model on Food-101 (Google Colab)

**What this notebook does:** transfer learning. It starts from a Vision Transformer that Google already pretrained on ImageNet-21k, and **fine-tunes it on the Food-101 dataset** (101 food classes, 75,750 training images, 25,250 test images). The result is *your own* food recognition model, plus a `training_report.json` with the accuracy you actually measured.

**How to describe it honestly:** "I fine-tuned a pretrained ViT on Food-101 using transfer learning." Do not say you built the ViT or pretrained it from scratch.

**Before you run:** Runtime > Change runtime type > **T4 GPU**. Training on a laptop CPU is not realistic.

**Quick test first:** keep `TRAIN_SUBSET = 3000` and `EPOCHS = 1` for a few-minute dry run. For the final model set `TRAIN_SUBSET = None` and `EPOCHS = 3`.

**Dataset note:** the task links the Kaggle copy (dansbecker/food-101). This notebook loads the same Food-101 data from the Hugging Face hub (`ethz/food101`), which avoids needing a Kaggle login. Mention that in your report.


In [1]:
# Do NOT upgrade torch or torchvision here. Colab's preinstalled versions already match each other.
!pip install -q -U transformers datasets accelerate
# torchaudio is not needed for images and can clash with torch's CUDA version.
!pip uninstall -y -q torchaudio


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
cudf-cu13 26.6.0 requires pyarrow<24,>=19.0.0, but you have pyarrow 26.0.0 which is incompatible.


## 0. Check the GPU is really on
If this cell raises an error, fix the runtime before going further. Training on CPU is far too slow.


In [2]:
import torch

print("PyTorch", torch.__version__, "| CUDA available:", torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError("No GPU found. Runtime > Change runtime type > T4 GPU, then Runtime > Restart session and run all cells again.")
print("GPU:", torch.cuda.get_device_name(0))


PyTorch 2.11.0+cu130 | CUDA available: True
GPU: Tesla T4


## 1. Settings

In [3]:
# ---- Settings you can change ----
BASE_MODEL = "google/vit-base-patch16-224-in21k"   # pretrained by Google, we fine-tune it
EPOCHS = 1              # use 3 for the final run
TRAIN_SUBSET = 3000     # None = use all 75,750 training images
EVAL_SUBSET = 2000      # None = use all 25,250 test images (slower)
BATCH_SIZE = 32
LEARNING_RATE = 5e-5
SEED = 42
OUT_DIR = "/content/nutrilens_food"


## 2. Load Food-101

In [4]:
from datasets import load_dataset

ds = load_dataset("ethz/food101")          # splits: train / validation (the official test set)
label_names = ds["train"].features["label"].names
id2label = {i: n for i, n in enumerate(label_names)}
label2id = {n: i for i, n in id2label.items()}
print(len(label_names), "classes;", len(ds["train"]), "train images;", len(ds["validation"]), "test images")

train_raw = ds["train"].shuffle(seed=SEED)
eval_raw = ds["validation"].shuffle(seed=SEED)
if TRAIN_SUBSET:
    train_raw = train_raw.select(range(min(TRAIN_SUBSET, len(train_raw))))
if EVAL_SUBSET:
    eval_raw = eval_raw.select(range(min(EVAL_SUBSET, len(eval_raw))))
print("Using", len(train_raw), "train and", len(eval_raw), "evaluation images")


README.md:   0%|          | 0.00/16.4k [00:00<?, ?B/s]

data/train-00000-of-00008.parquet: reconstructing file:   0%|          |  0.00B /  490MB            

data/train-00000-of-00008.parquet: downloading bytes:           |  0.00B            

data/train-00001-of-00008.parquet: reconstructing file:   0%|          |  0.00B /  464MB            

data/train-00001-of-00008.parquet: downloading bytes:           |  0.00B            

data/train-00002-of-00008.parquet: reconstructing file:   0%|          |  0.00B /  472MB            

... (16 outputs left out)

data/validation-00002-of-00003.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/75750 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/25250 [00:00<?, ? examples/s]

101 classes; 75750 train images; 25250 test images
Using 3000 train and 2000 evaluation images


## 3. Preprocessing and augmentation

In [5]:
import numpy as np
import torch
from torchvision.transforms import CenterCrop, Compose, Normalize, RandomHorizontalFlip, RandomResizedCrop, Resize, ToTensor
from transformers import AutoImageProcessor

processor = AutoImageProcessor.from_pretrained(BASE_MODEL)
size = processor.size["height"] if "height" in processor.size else processor.size["shortest_edge"]
normalize = Normalize(mean=processor.image_mean, std=processor.image_std)

train_tf = Compose([RandomResizedCrop(size), RandomHorizontalFlip(), ToTensor(), normalize])   # augmentation
eval_tf = Compose([Resize(size), CenterCrop(size), ToTensor(), normalize])

def make_transform(tf):
    def apply(batch):
        batch["pixel_values"] = [tf(img.convert("RGB")) for img in batch["image"]]
        del batch["image"]
        return batch
    return apply

train_ds = train_raw.with_transform(make_transform(train_tf))
eval_ds = eval_raw.with_transform(make_transform(eval_tf))

def collate(examples):
    return {
        "pixel_values": torch.stack([e["pixel_values"] for e in examples]),
        "labels": torch.tensor([e["label"] for e in examples]),
    }

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    return {"accuracy": float((np.argmax(logits, axis=-1) == labels).mean())}


preprocessor_config.json:   0%|          | 0.00/160 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/502 [00:00<?, ?B/s]

## 4. Fine-tune

In [6]:
from transformers import AutoModelForImageClassification, Trainer, TrainingArguments

model = AutoModelForImageClassification.from_pretrained(
    BASE_MODEL, num_labels=len(label_names), id2label=id2label, label2id=label2id
)

args = TrainingArguments(
    output_dir="/content/checkpoints",
    remove_unused_columns=False,
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=LEARNING_RATE,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    num_train_epochs=EPOCHS,
    weight_decay=0.01,
    fp16=torch.cuda.is_available(),
    logging_steps=50,
    save_total_limit=1,
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    dataloader_num_workers=2,
    report_to="none",
    seed=SEED,
)

trainer = Trainer(
    model=model,
    args=args,
    data_collator=collate,
    train_dataset=train_ds,
    eval_dataset=eval_ds,
    compute_metrics=compute_metrics,
    processing_class=processor,
)
trainer.train()


model.safetensors: reconstructing file:   0%|          |  0.00B /  346MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

[transformers] ViTForImageClassification LOAD REPORT from: google/vit-base-patch16-224-in21k
Key                 | Status     | 
--------------------+------------+-
pooler.dense.weight | UNEXPECTED | 
pooler.dense.bias   | UNEXPECTED | 
classifier.weight   | MISSING    | 
classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch,Training Loss,Validation Loss,Accuracy
1,4.557878,4.438691,0.211000


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

[transformers] There were missing keys in the checkpoint model loaded: ['vit.layers.0.attention.q_proj.weight', 'vit.layers.0.attention.q_proj.bias', 'vit.layers.0.attention.k_proj.weight', 'vit.layers.0.attention.k_proj.bias', 'vit.layers.0.attention.v_proj.weight', 'vit.layers.0.attention.v_proj.bias', 'vit.layers.0.attention.o_proj.weight', 'vit.layers.0.attention.o_proj.bias', 'vit.layers.0.layernorm_before.weight', 'vit.layers.0.layernorm_before.bias', 'vit.layers.0.layernorm_after.weight', 'vit.layers.0.layernorm_after.bias', 'vit.layers.0.mlp.fc1.weight', 'vit.layers.0.mlp.fc1.bias', 'vit.layers.0.mlp.fc2.weight', 'vit.layers.0.mlp.fc2.bias', 'vit.layers.1.attention.q_proj.weight', 'vit.layers.1.attention.q_proj.bias', 'vit.layers.1.attention.k_proj.weight', 'vit.layers.1.attention.k_proj.bias', 'vit.layers.1.attention.v_proj.weight', 'vit.layers.1.attention.v_proj.bias', 'vit.layers.1.attention.o_proj.weight', 'vit.layers.1.attention.o_proj.bias', 'vit.layers.1.layernorm_before

TrainOutput(global_step=94, training_loss=4.518447713649019, metrics={'train_runtime': 83.4379, 'train_samples_per_second': 35.955, 'train_steps_per_second': 1.127, 'total_flos': 2.326822460928e+17, 'train_loss': 4.518447713649019, 'epoch': 1.0})

## 5. Evaluate and save (the accuracy printed here is the number to quote)

In [7]:
import json, shutil
from datetime import date

metrics = trainer.evaluate()
accuracy = metrics["eval_accuracy"]
print(f"Accuracy on {len(eval_raw)} held-out test images: {accuracy:.2%}")

trainer.save_model(OUT_DIR)
processor.save_pretrained(OUT_DIR)

report = {
    "base_model": BASE_MODEL,
    "dataset": "Food-101 (ethz/food101 on Hugging Face, official train/test split)",
    "train_images": len(train_raw),
    "eval_images": len(eval_raw),
    "epochs": EPOCHS,
    "validation_accuracy": round(accuracy, 4),
    "trained_on": "Google Colab",
    "date": str(date.today()),
}
with open(f"{OUT_DIR}/training_report.json", "w") as f:
    json.dump(report, f, indent=2)
print(json.dumps(report, indent=2))


Training Loss,Validation Loss,Epoch,Accuracy
4.557878,4.438691,1,0.211000


Accuracy on 2000 held-out test images: 21.10%


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

{
  "base_model": "google/vit-base-patch16-224-in21k",
  "dataset": "Food-101 (ethz/food101 on Hugging Face, official train/test split)",
  "train_images": 3000,
  "eval_images": 2000,
  "epochs": 1,
  "validation_accuracy": 0.211,
  "trained_on": "Google Colab",
  "date": "2026-11-02"
}


## 6. Sanity check on a few test images

In [8]:
from transformers import pipeline

clf = pipeline("image-classification", model=OUT_DIR, device=0 if torch.cuda.is_available() else -1)
for i in range(3):
    sample = ds["validation"][i * 1000]
    print("true:", label_names[sample["label"]], "->", [(r["label"], round(r["score"], 3)) for r in clf(sample["image"].convert("RGB"), top_k=3)])


Loading weights:   0%|          | 0/200 [00:00<?, ?it/s]

true: beignets -> [('sushi', 0.013), ('french_toast', 0.013), ('pho', 0.012)]
true: bruschetta -> [('pulled_pork_sandwich', 0.014), ('beet_salad', 0.014), ('foie_gras', 0.013)]
true: sashimi -> [('lobster_roll_sandwich', 0.014), ('sushi', 0.013), ('tuna_tartare', 0.013)]


## 7. Download your model

In [9]:
shutil.make_archive("/content/nutrilens_food", "zip", OUT_DIR)
from google.colab import files
files.download("/content/nutrilens_food.zip")   # about 330 MB; may take a minute


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## After the download
1. Unzip `nutrilens_food.zip`.
2. Put the unzipped files in your project at `NutriLens/models/nutrilens_food/` (the folder must contain `config.json`, the weights file, and `training_report.json`).
3. Restart Streamlit. The app detects the folder and uses **your** model. It shows the accuracy from `training_report.json`.

If the folder is missing, the app falls back to the pretrained `yfh/food` model and says so.
